In [ ]:
import sys
from pathlib import Path

import os
import sys
from pathlib import Path
import json
import warnings
import shutil
import random
import string

import torch
from torch.nn import functional as F
import nibabel as nib
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, r2_score
from scipy.stats import pearsonr, spearmanr

from lit_classifier import LitClassifier
from datasets import NiftiDataset, nifti_loader
from utils import confusion_matrix_to_figure
from transformations import get_transforms
from utils_hparams import args_short_to_long

warnings.filterwarnings("ignore")

seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True  # CuDNN deterministic
torch.backends.cudnn.benchmark = False  # CuDNN benchmark off
torch.random.manual_seed(seed)
pl.seed_everything(seed)

In [ ]:
## resnet_film_sigmoid
best_weights_path = '$PATH/WM_ICV/log/basic_0519_ESNR/ratio_origin_15/version_3/checkpoints/epoch=085.ckpt'


def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 126
device = torch.device('cuda:9')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data
sample = samples[6]
print(sample.shape, y)

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam_info import GradCAM
from pytorch_grad_cam_info.utils.image import show_cam_on_image

encoder = model.backbone
layer_names = ['conv1', 'conv2', 'conv3']
layers = [encoder.layer1[0].conv1, encoder.layer2[0].conv1, encoder.layer3[0].conv1]


for i in range(126):
    if y[i] <= 10:
        print(y[i])
    else:
        print(y[i])
        sample = samples[i]
        wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
        icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
        info = torch.concat((wm, icv), dim=1)
        slice_idx = 40
        input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
        input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

        img_slice = input_for_viz[:, :, slice_idx, 0:1]
        img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

        # figure setup
        fig, axes = plt.subplots(1, len(layers) + 1, figsize=(3 * (len(layers) + 1), 4))

        fig.text(0.02, 0.5, 'RI', va='center', rotation='vertical', fontsize=25)

        axes[0].imshow(np.rot90(img_norm), cmap='gray')
        axes[0].axis("off")

        for i, layer in enumerate(layers):
            cam = GradCAM(model=model, target_layers=[layer])
            grayscale_cam = cam(input_tensor=input_tensor, infos=info, targets=None)[0]  # [D, H, W]
            cam_slice = grayscale_cam[:, :, slice_idx]
            overlay = show_cam_on_image(img_norm, cam_slice, use_rgb=True, image_weight=0.05)

            axes[i + 1].imshow(np.rot90(overlay), cmap='jet')
            axes[i + 1].axis("off")

        plt.tight_layout(rect=[0.04, 0, 1, 1])
        plt.show()
        plt.close() 